# Visualising Bunny Outputs
There are two examples of outputs from a Bunny distribution query included in the test data.
This kind of data can be obtained from Five Safes TES (5STES) by submitting a TES message as follows:

```json
{
         "id": "someID",
         "state": 0,
         "name": "Bunny testing",
         "description": null,
         "inputs": null,
         "outputs": [
                  {
                           "name": "Query Results",
                           "description": "Results from the requested query execution",
                           "url": "s3://",
                           "path": "/outputs",
                           "type": "DIRECTORY"
                  }
         ],
         "resources": null,
         "executors": [
                  {
                           "image": "ghcr.io/health-informatics-uon/five-safes-tes-analytics-bunny-cli:1.6.0",
                           "command": [
                                    "--body-json",
                                    "{\"code\":\"GENERIC\",\"analysis\":\"DISTRIBUTION\",\"uuid\":\"123\",\"collection\":\"test\",\"owner\":\"me\"}",
                                    "--output",
                                    "/outputs/output.json",
                                    "--no-encode"
                           ],
                           "workdir": null,
                           "stdin": null,
                           "stdout": null,
                           "stderr": null,
                           "env": null
                  }
         ],
         "volumes": null,
         "tags": {
                  "project": "someProject",
                  "tres": "someTREs"
         },
         "logs": null,
         "creation_time": null
}

```

A message like this can be submitted using the submission layer's Raw JSON wizard, substituting your own parameters for `"id"`, `"tags"/"project"` and `"tags/tres"`.

This kind of TES message can also be created by using the custom image wizard:
![Screenshot of the custom image wizard running a bunny query](./bunny-custom-image-wizard.png)

The settings need to be:

- **Docker Image**: ghcr.io/health-informatics-uon/five-safes-tes-analytics-bunny-cli:latest
- **Commands**:
```
--body-json
{"code":"GENERIC","analysis":"DISTRIBUTION","uuid":"123","collection":"test","owner":"me"}
--output
/outputs/output.json
--no-encode
```

We have provided some utilities to help users interpret the outputs of these distribution queries, importable from `bunny_utils`.

In [1]:
from five_safes_tes_analytics.utils.parse_bunny import parse_bunny
from bunny_utils import DistributionCodesets, count_bar
import warnings
warnings.filterwarnings('ignore')

The examples "1k TRE" and "100k TRE" are taken from the synthetic OMOP datasets held in University of Nottingham test TREs.
The other examples are dummy data created for this demonstration.
Running the wizard, you should be able to egress files for your output.
Hopefully, you've kept track of which files come from which TRE.

Initialising a `DistributionCodesets` object with a dictionary with names you recognise and the paths to the files means that visualisations etc. will keep those labels.

In [2]:
bunny_table_names = {
    "1k TRE": "../tests/test-data/1kconcepts.json",
    "100k TRE": "../tests/test-data/100kconcepts.json",
    "Narnia": "bunny-dummy-data/narnia-tsv-dummy.json",
    "The Moon": "bunny-dummy-data/the-moon-tsv-dummy.json",
    "Tingham": "bunny-dummy-data/tingham-tsv-dummy.json"
}

In [3]:
codesets = DistributionCodesets(bunny_table_names)

You can look at the raw tables from the query in the `.tables` attribute.

In [4]:
codesets.tables["1k TRE"].head()

,TRE,BIOBANK,CODE,COUNT,ALTERNATIVES,DATASET,OMOP,OMOP_DESCR,CATEGORY
0,1k TRE,test,OMOP:0,580,NaN,NaN,0,No matching concept,Condition
1,1k TRE,test,OMOP:28060,150,NaN,NaN,28060,Streptococcal sore throat,Condition
2,1k TRE,test,OMOP:75036,20,NaN,NaN,75036,"Localized, primary osteoarthritis of the hand",Condition
3,1k TRE,test,OMOP:78272,50,NaN,NaN,78272,Sprain of wrist,Condition
4,1k TRE,test,OMOP:80502,50,NaN,NaN,80502,Osteoporosis,Condition


You can get the counts for each code on each TRE with the `counts_by_TRE` property.

In [5]:
codesets.counts_by_TRE.head()

TRE,100k TRE,1k TRE,Narnia,The Moon,Tingham
OMOP,,,,,
8507,49740.0,570.0,NaN,NaN,NaN
8515,1920.0,70.0,NaN,NaN,NaN
8516,2020.0,80.0,NaN,NaN,NaN
8527,2020.0,840.0,NaN,NaN,NaN
8532,50260.0,560.0,NaN,NaN,NaN


You can view how many codes your TREs have in common with the `code_intersections` property.
This example shows that the 100k and 1k TREs share 7 codes, that "100k TRE" has 8885 unique codes, and the "1k TRE" has 361 unique codes.

In [6]:
codesets.code_intersections

,0
"['100k TRE', '1k TRE']",7
"['100k TRE', 'Narnia', 'The Moon', 'Tingham']",16
"['100k TRE', 'Narnia', 'The Moon']",1
"['100k TRE', 'Tingham']",3
['100k TRE'],8865
"['1k TRE', 'Narnia', 'The Moon', 'Tingham']",2
['1k TRE'],359
"['Narnia', 'The Moon', 'Tingham']",1056
"['Narnia', 'The Moon']",925
['Tingham'],921


You can plot the $k$ codes with the highest counts using `.plot_top_k_by_count(k)`.
If you run this notebook, you can hover over the bars to get the OMOP description of that code.

In [7]:
codesets.plot_top_k_by_count(10).

alt.Chart(...)

If you have codes that you're interested in, you can use the `.plot_by_codes(list_of_codes)` method to get a barplot of those.

In [8]:
codesets.plot_by_codes([28060, 3000905])

alt.Chart(...)

You can combine this method with ways of generating lists of codes, for example, getting the list of codes that are shared between some TREs, using the `.get_codes_by_membership` method.
These are the codes shared by both of the synthetic datasets.

In [9]:
codesets.plot_by_codes(codesets.get_codes_by_membership("['100k TRE', '1k TRE']")["OMOP"])

alt.Chart(...)

Or if you don't have a set of codes you want to query, but do have some substring to match, you can use the `get_codes_by_substring_match` method. This is case-insensitive and supports regular expressions.

In [10]:
codesets.plot_by_codes(codesets.get_codes_by_substring_match("leuko")["OMOP"])

alt.Chart(...)

You can get a heatmap of how many codes are in each combination of datasets as a heatmap with `.plot_count_heatmap`

In [11]:
codesets.plot_count_heatmap().properties(width=600, height=600)

alt.Chart(...)

You can also get an [Upset plot](https://upset.app/) for your datasets.
This is like a Venn diagram, but instead of numbers written in circles, you get bars proportional to the number of codes present in each combination of TREs.
This shows the same information as the `code_insersections` property.
If you only have a couple of TREs, this isn't terribly useful, but once you have more than three, the number of combinations is much higher, and Venn diagrams get hard to read.

In [12]:
codesets.plot_upset()

alt.VConcatChart(...)